# 00. Research protocol

Fix the accounting objects, the source regimes and the non-interpretive scope before any result is inspected. Nothing in this notebook depends on a finding, so the analytical rules cannot be adjusted after seeing the numbers.

**Reads**

- `config/sources.yml`
- `outputs/metrics/raw_file_sha256.json`

**Writes**

- Nothing. This notebook states the protocol and inspects source metadata.

**Method reference:** `METHODOLOGY.md` sections 1-3 and 16

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. The object of study

The primary measure is B.9, net lending (+) / net borrowing (-), for the calendar
years 1977 to 2025. The canonical decomposition is

$$B^{GG}_t = B^{C}_t + B^{RL}_t + B^{SSF}_t,$$

where the three components are Central Government, Regional and Local Government,
and Social Security Funds.

The study describes accounting contributions, temporal dynamics and statistical
relationships. It does not infer intent, assign responsibility, or attach normative
labels to fiscal outcomes. Where a metric could invite such a reading, the notebook
that produces it states what the metric is not.

## 2. Sources under version control

Each source is pinned to a local file in `data/raw`. The workbooks are parsed
programmatically; no value is transcribed by hand.

In [ ]:
import yaml

config = yaml.safe_load((ROOT / 'config' / 'sources.yml').read_text(encoding='utf-8'))
sources = pd.DataFrame(config['sources']).T[['institution', 'coverage', 'local_file']]
display(sources)

## 3. Raw-source integrity

Every bundled raw file is hashed by the pipeline. A reader who obtains the same
files can confirm they are analysing identical inputs.

In [ ]:
import json

hashes = json.loads((METRICS / 'raw_file_sha256.json').read_text(encoding='utf-8'))
manifest = pd.DataFrame(
    [
        {
            'file': path,
            'size_kb': round((ROOT / path).stat().st_size / 1024, 1),
            'sha256_prefix': digest[:16],
        }
        for path, digest in sorted(hashes.items())
    ]
)
display(manifest)

## 4. Data-layer contract

```text
data/raw/         official source files, immutable in normal use
data/interim/     source-specific extractions and overlap checks
data/processed/   canonical analysis-ready panels
outputs/tables/   calculated tables and statistical results
outputs/metrics/  validation summaries and source hashes
outputs/figures/  figures generated from processed results
report/           report generated from the persisted outputs only
```

A result may only enter the report if it was first written to one of those
layers. That rule is what makes the report checkable independently of the code
that produced it.

In [ ]:
layers = pd.DataFrame(
    [
        {'layer': label, 'files': len(list((ROOT / relative).glob(pattern)))}
        for label, relative, pattern in [
            ('data/raw', 'data/raw', '**/*.*'),
            ('data/interim', 'data/interim', '*.csv'),
            ('data/processed', 'data/processed', '*.csv'),
            ('outputs/tables', 'outputs/tables', '*.csv'),
            ('outputs/metrics', 'outputs/metrics', '*.json'),
            ('outputs/figures', 'outputs/figures', '*.png'),
        ]
    ]
)
display(layers)

## 5. The notebook sequence

The notebooks are the visible narrative and run in lexical order: extraction,
harmonisation and validation, then one notebook per analytical question, and
finally report generation.

In [ ]:
stages = []
for path in sorted((ROOT / 'notebooks').glob('*.ipynb')):
    cells = json.loads(path.read_text(encoding='utf-8'))['cells']
    heading = next(
        line
        for cell in cells
        if cell['cell_type'] == 'markdown'
        for line in cell['source']
        if line.startswith('# ')
    )
    stages.append({'notebook': path.name, 'stage': heading.removeprefix('# ').strip()})
display(pd.DataFrame(stages))

## Interpretation limits

1. **1995 is a source and methodology splice**, not an economic event. Both
   vintages of 1995 are retained as a diagnostic and nothing is smoothed across
   the boundary.
2. **Detailed subsector accounts have no 1996-1999 observations.** The gap is
   left explicit rather than interpolated.
3. **Accounting identities are not causal statements.** A subsector that
   contributes arithmetically to an aggregate balance has not been shown to have
   caused it.
4. **Descriptive regressions are co-movement estimates.** Nominal GDP growth is
   not an output gap and no specification here is a structural fiscal model.

---

[Next: 01. Historical extraction: Banco de Portugal / INE](01_extract_historical_data.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```